In [ ]:
import contextlib,ctypes,hashlib,importlib.metadata,json,os,signal,subprocess,sys,time,urllib.request
from pathlib import Path
from unittest.mock import patch
ROOT=Path('/kaggle/working');CACHE=Path('/tmp')/('mgbfs-prefetch-shared-'+str(os.getpid()));WORK=CACHE.parent/(CACHE.name+'-work');WORK.mkdir()
BASE='https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261015-v20/'
m=json.loads(urllib.request.urlopen(BASE+'manifest.json',timeout=60).read());assert m['sha256']=='2a3213f46e02256dc3eeddf094a80488ceeddaba2f2bc70cebb85ee6c69ba7cb' and m['size']<2_000_000
wheel=Path('/tmp')/m['wheel'];urllib.request.urlretrieve(BASE+m['wheel'],wheel);assert hashlib.sha256(wheel.read_bytes()).hexdigest()==m['sha256']
subprocess.run([sys.executable,'-m','pip','install','--force-reinstall','--no-deps',str(wheel)],check=True)
assert importlib.metadata.version('multigpubfs')=='0.2.0.dev20'
raw=subprocess.check_output(['nvidia-smi','--query-gpu=name,compute_cap,driver_version','--format=csv,noheader'],text=True)
hardware=[{'name':s.split(',')[0].strip(),'cc':[int(v) for v in s.split(',')[1].strip().split('.')],'driver':s.split(',')[2].strip()} for s in raw.splitlines() if s.strip()]
assert len(hardware)==2 and all(x['cc']==[7,5] for x in hardware),hardware
receipt={'schema':1,'status':'RUNNING','package':'0.2.0.dev20','wheel_sha256':m['sha256'],'wheel_bytes':m['size'],'package_source_commit':m['source_commit'],'native_index_sha256':m['native_index_sha256'],'hardware':hardware,'checks':[],'scope':'Actual prefetch with GPU access forbidden, then physical 2xT4 offline BFS. Other SM files are hash-verified, not GPU execution acceptance.'}
started=time.monotonic()
def record(name,**data):
 receipt['checks'].append(dict(name=name,status='PASS',**data));(ROOT/'prefetch-gpu-acceptance.json').write_text(json.dumps(receipt,indent=2));print('PREFETCH_GPU_CHECK',json.dumps(receipt['checks'][-1]),flush=True)
import multigpubfs
from multigpubfs import native_assets as assets
os.environ['MGBFS_NATIVE_CACHE']=str(CACHE)
def forbidden(*a,**k):raise AssertionError('GPU-free prefetch attempted driver/library/process access')
before=time.monotonic()
with patch.dict(os.environ,{'CUDA_VISIBLE_DEVICES':''}),patch.object(subprocess,'Popen',side_effect=forbidden),patch.object(ctypes,'CDLL',side_effect=forbidden),patch.object(ctypes,'PyDLL',side_effect=forbidden),patch.object(assets,'run_hardware_probe',side_effect=forbidden):
 result=multigpubfs.prefetch_native(sm=[75,100],cache=CACHE)
elapsed=time.monotonic()-before
index=json.loads(assets.index_path().read_text());names={index['common'],index['production']['75'],index['production']['100']}
assert result['status']=='PREFETCHED' and result['downloaded_bytes']==sum(index['components'][n]['size'] for n in names)
assert result['package_source_commit']==m['source_commit'] and assets.status()['status']=='NOT_PREPARED'
record('selected-prefetch-before-gpu-allocation',targets=result['targets'],downloaded_bytes=result['downloaded_bytes'],seconds=elapsed,gpu_access_forbidden=True,components=[{'component':x['component'],'sha256':x['sha256']} for x in result['transfers']])
os.environ['MGBFS_NATIVE_OFFLINE']='1'
q=subprocess.run([sys.executable,'-m','multigpubfs','prefetch','--sm','75,100','--json'],env=dict(os.environ,CUDA_VISIBLE_DEVICES=''),capture_output=True,text=True,check=True,timeout=90)
warm=json.loads(q.stdout);assert warm['status']=='PREFETCHED' and warm['downloaded_bytes']==0 and warm['maximum_additional_download_bytes']==0
record('fresh-process-cli-prefetch-offline-no-visible-gpu',downloaded_bytes=0)

child=r"""import hashlib,json,os,sys
from pathlib import Path
from multigpubfs import native_assets as assets
from multigpubfs.graph_definition import GraphDefinition
from multigpubfs.launch import run_graph
out=Path(sys.argv[1]);g=GraphDefinition.permutation([[1,0,2],[0,2,1]],[0,1,2])
r=run_graph(g,out,devices=[0],capacity=64,shards=1,autotune=False,max_seconds=60)
assert r['status']=='COMPLETE' and r['layer_sizes']==[1,2,2,1]
s=assets.status();assert s['selected_component']=='cuda-sm75' and s['downloaded_bytes']==0 and len(s['hardware']['devices'])==1
states=json.loads((out/'states.json').read_text());assert states['current']==[[2,1,0]]
proof={'status':'PASS','visible_physical_gpu':os.environ['CUDA_VISIBLE_DEVICES'],'logical_devices':r['devices'],'layers':r['layer_sizes'],'bfs_seconds':r['bfs_seconds'],'selected_component':s['selected_component'],'downloaded_bytes':s['downloaded_bytes'],'library_sha256':r['runtime_identity']['cuda_library_sha256'],'pid':os.getpid()}
Path(sys.argv[2]).write_text(json.dumps(proof));print('OFFLINE_GPU_JOB_PASS',json.dumps(proof),flush=True)
"""
jobs=[]
try:
 for physical in (0,1):
  output=WORK/('parallel-'+str(physical));proof=ROOT/('parallel-'+str(physical)+'.json')
  p=subprocess.Popen([sys.executable,'-c',child,str(output),str(proof)],env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(physical)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,start_new_session=True);jobs.append((p,proof))
 for p,proof in jobs:
  log,_=p.communicate(timeout=150);print(log[-12000:],flush=True);assert p.returncode==0,p.returncode
 proofs=[json.loads(proof.read_text()) for p,proof in jobs]
 assert {x['visible_physical_gpu'] for x in proofs}=={'0','1'} and len({x['pid'] for x in proofs})==2
 record('parallel-two-physical-gpus-shared-offline-cache',jobs=proofs,downloaded_bytes=0)
finally:
 for p,proof in jobs:
  if p.poll() is None:
   os.killpg(p.pid,signal.SIGTERM)
   try:p.wait(timeout=2)
   except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait(timeout=2)

from multigpubfs.graph_definition import GraphDefinition
from multigpubfs.launch import run_graph
from multigpubfs.generic_session import persistent_graph_workers
g=GraphDefinition.permutation([[1,2,3,0],[3,0,1,2],[1,0,2,3]],[0,1,2,3])
frontier={(0,1,2,3)};seen=set(frontier);layers=[]
while frontier:
 layers.append(len(frontier));terminal=set(frontier)
 children={tuple(state[i] for i in generator) for state in frontier for generator in ((1,2,3,0),(3,0,1,2),(1,0,2,3))}
 frontier=children-seen;seen.update(frontier)
previous=None
with patch.object(urllib.request,'urlopen',side_effect=AssertionError('Offline BFS accessed network')),persistent_graph_workers() as pool:
 for repeat in (1,2):
  out=WORK/('two-gpu-'+str(repeat));r=run_graph(g,out,devices=[0,1],capacity=128,shards=1,autotune=False,max_seconds=60)
  assert r['status']=='COMPLETE' and r['layer_sizes']==layers and r['devices']==[0,1]
  states=json.loads((out/'states.json').read_text());assert sorted(tuple(row) for row in states['current'])==sorted(tuple(row) for row in terminal)
  pids=[w.process.pid for w in pool.workers];assert len(pids)==2 and (previous is None or previous==pids);previous=pids
  s=assets.status();assert s['selected_component']=='cuda-sm75' and s['downloaded_bytes']==0
  record('two-gpu-persistent-offline-'+str(repeat),devices=r['devices'],layer_sizes=layers,terminal_states_cpu_verified=True,bfs_seconds=r['bfs_seconds'],worker_pids=pids,downloaded_bytes=0)
unused=index['components'][index['production']['100']];file=CACHE/'components'/unused['sha256']/'libmgbfs_cuda.so'
assert hashlib.sha256(file.read_bytes()).hexdigest()==unused['files']['libmgbfs_cuda.so']['sha256']
record('other-sm-cache-preserved-after-gpu-jobs',component='cuda-sm100',hash_verified=True)
receipt.update(status='PASS',wall_seconds=time.monotonic()-started);(ROOT/'prefetch-gpu-acceptance.json').write_text(json.dumps(receipt,indent=2));print('PREFETCH_GPU_ACCEPTANCE_PASS',json.dumps(receipt),flush=True)
